In [1]:
"""
CS4771 - Python for Machine Learning
Kaggle Competition Assignment 3
Todd Carter
V01187982
11-17-2025

"""


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import pandas as pd
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Hyperparameters ---
LEARNING_RATE = 0.001
BATCH_SIZE = 64
EPOCHS = 10

sample = pd.read_csv("/kaggle/input/u-of-idaho-f-25-pml-competition-3/sample_submission.csv")

# Print headers to see the data types:
print("\sample header is: \n", sample.head())

data_dir = "/kaggle/input/u-of-idaho-f-25-pml-competition-3"

# Transforms set with random parameters
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# A custom dataset class for images
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, dir, transform=None):
        self.data_dir = dir
        self.images = sorted(os.listdir(dir))
        self.transform = transform

    # Defining the length of the dataset
    def __len__(self):
        return len(self.images)

    # Defining the method to get an item from the dataset
    def __getitem__(self, index):
        image_name = self.images[index]
        
        image_path = os.path.join(self.data_dir, self.images[index])
        image = Image.open(image_path).convert("RGB")

        # Applying the transform
        if self.transform:
            image = self.transform(image)

        img_id = int(os.path.splitext(image_name)[0])
        
        return image, img_id

train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=transform)
test_dataset = ImageDataset("/kaggle/input/u-of-idaho-f-25-pml-competition-3/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# A relatively smooth but simple CNN class
class FRUIT_CNN(nn.Module):
    def __init__(self):
        super(FRUIT_CNN, self).__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create an instance of the model
model = FRUIT_CNN()

# Activate the cuda cores if available
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

# Move the model's parameters to the selected device (GPU or CPU)
model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Loop for the specified number of epochs
for epoch in range(EPOCHS):
    # Set the model to training mode
    # This is important for layers like Dropout or BatchNorm
    model.train()

    # Loop over the training data in batches
    # batch_idx provides a running count
    # (data, target) is a batch from the train_loader
    for batch_idx, (data, target) in enumerate(train_loader):
        # 1. Move data and target tensors to the device (GPU/CPU)
        data, target = data.to(device), target.to(device)

        # 2. Zero the gradients
        # We need to do this on every batch because PyTorch
        # accumulates gradients by default.
        optimizer.zero_grad()

        # 3. Forward Pass
        # Pass the data through the model
        output = model(data)

        # 4. Calculate Loss
        # Compare the model's output with the true target labels
        loss = criterion(output, target)

        # 5. Backward Pass
        # Compute the gradients of the loss w.r.t. model parameters
        loss.backward()

        # 6. Update Weights
        # Tell the optimizer to adjust the weights based on the gradients
        optimizer.step()

        # Print training status
        if batch_idx % 100 == 0:
            print(f"Epoch: {epoch+1}/{EPOCHS} | Batch: {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")
    torch.save(model.state_dict(), f"saved_model.pt")


# Set the model to evaluation mode
model.eval()

# Two lists to collect data for submission
predictions = []
IDs = []

# Disable gradient calculations to save memory and compute
with torch.no_grad():
    # Loop over the test data
    for data, ID in test_loader:
        data = data.to(device)

        # Get model output
        output = model(data)

        # Get the index of the max log-probability (our prediction)
        pred = output.argmax(dim=1)

        # Load the two lists with data for submission
        predictions.extend(pred.cpu().numpy())
        IDs.extend(ID.cpu().numpy())


# Produce the submission csv:
submission = pd.DataFrame({
    "id": [int(x) for x in IDs],
    "label": [int(x) for x in predictions]
})

submission.to_csv("submission.csv", index=False)
print("\nSubmission created!")

Using device: cuda
\sample header is: 
    id  label
0   0      0
1   1      0
2   2      0
3   3      0
4   4      0
Epoch: 1/10 | Batch: 0/555 | Loss: 1.9905
Epoch: 1/10 | Batch: 100/555 | Loss: 0.6601
Epoch: 1/10 | Batch: 200/555 | Loss: 0.2990
Epoch: 1/10 | Batch: 300/555 | Loss: 0.2087
Epoch: 1/10 | Batch: 400/555 | Loss: 0.1476
Epoch: 1/10 | Batch: 500/555 | Loss: 0.1056
Epoch: 2/10 | Batch: 0/555 | Loss: 0.1995
Epoch: 2/10 | Batch: 100/555 | Loss: 0.1165
Epoch: 2/10 | Batch: 200/555 | Loss: 0.1572
Epoch: 2/10 | Batch: 300/555 | Loss: 0.1389
Epoch: 2/10 | Batch: 400/555 | Loss: 0.1347
Epoch: 2/10 | Batch: 500/555 | Loss: 0.0578
Epoch: 3/10 | Batch: 0/555 | Loss: 0.0985
Epoch: 3/10 | Batch: 100/555 | Loss: 0.0543
Epoch: 3/10 | Batch: 200/555 | Loss: 0.1050
Epoch: 3/10 | Batch: 300/555 | Loss: 0.0757
Epoch: 3/10 | Batch: 400/555 | Loss: 0.0331
Epoch: 3/10 | Batch: 500/555 | Loss: 0.0283
Epoch: 4/10 | Batch: 0/555 | Loss: 0.0186
Epoch: 4/10 | Batch: 100/555 | Loss: 0.1138
Epoch: 4/1